# Geometry Working Memory

https://www.biorxiv.org/content/10.64898/2026.08.31.748237v1

In [ ]:
# libraries

import os
import numpy as np
import pickle
import scipy


In [38]:
# functions

def compute_pca(X):

    """
    Compute PCA through eigendecomposition of covariance matrix
    pc_scores[:,0:k]: Leading k pc_scores
    eigvecs[:,0:k]:   Leading k eigengectors

    """

    # Step 1: Center the data (subtract the mean of each feature)
    X_mean = np.mean(X, axis=0)
    X_centered = X - X_mean

    # Step 2: Compute the covariance matrix
    cov_matrix = np.cov(X_centered, rowvar=False)

    # Step 3: Eigenvalue decomposition using eigh
    eigvals, eigvecs = scipy.linalg.eigh(cov_matrix)

    # Sort the eigenvectors by eigenvalues (descending order)
    sorted_indices = np.argsort(eigvals)[::-1]
    eigvals = eigvals[sorted_indices]
    eigvecs = eigvecs[:, sorted_indices]

    # Step 4: Compute principal component scores
    pc_scores = np.dot(X_centered, eigvecs)

    return eigvecs, eigvals, pc_scores



In [ ]:
### set paths and settings

path_root = '/path_to_local'

# settings

subjects = [f'sub_{i:02d}' for i in range(1, 50)]

time_windows = ['encode', 'maint', 's2']
pca_folder = 'pca_noaligned'
folder_input = 'X_matrix'
number_components_align = 10

outputs_correct_trials = [
'control_2gratings_2polygons',                                  'control_1gratings_1polygons',
'update_2gratings_relevant_2polygons_nonrelevant',              'update_1gratings_relevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_relevant',              'update_1gratings_nonrelevant_1polygons_relevant',
'inhibition_2gratings_relevant_2polygons_nonrelevant',          'inhibition_1gratings_relevant_1polygons_nonrelevant',
'inhibition_2gratings_nonrelevant_2polygons_relevant',        'inhibition_1gratings_nonrelevant_1polygons_relevant',
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      'update_1gratings_nolongerrelevant_1polygons_nonrelevant',
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',      'update_1gratings_nonrelevant_1polygons_nolongerrelevant',
]

outputs_incorrect_trials = [
'control_2gratings_2polygons',                                  
'update_2gratings_relevant_2polygons_nonrelevant',              
'update_2gratings_nonrelevant_2polygons_relevant',              
'inhibition_2gratings_relevant_2polygons_nonrelevant',          
'inhibition_2gratings_nonrelevant_2polygons_relevant',        
'update_2gratings_nolongerrelevant_2polygons_nonrelevant',      
'update_2gratings_nonrelevant_2polygons_nolongerrelevant',       
]


# Principal Component Analysis - correct trials

In [40]:
trial_type = 'correct_trials'

if trial_type == 'correct_trials':
    outputs = outputs_correct_trials
elif trial_type == 'incorrect_trials':
    outputs = outputs_incorrect_trials

for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    for sub_i in subjects:

        # paths
        path_X_sub = os.path.join(path_root, 'results', folder_input, time_window + '_time_resolved', sub_i)
        path_pca_sub = os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)

        if not os.path.isdir(path_pca_sub):
            os.makedirs(path_pca_sub)

        # load X matrices
        filename = os.path.join(path_X_sub, 'X_dict_' + trial_type + '.pkl')
        with open(filename, 'rb') as file:
            X_dict = pickle.load(file)

        pca_dict = {}

        for time_segment in segments:
            
            # loop over outputs                    
            for output in outputs:
                    
                key = time_window + '_segment' + str(time_segment) + '_' + output

                # load X matrix
                X = X_dict[key]
                                                    
                # pca
                eigvecs, eigvals, pc_scores = compute_pca(X)

                # store
                pca_dict['eigvecs_' + key] = eigvecs[:,0:number_components_align]
                pca_dict['eigvals_' + key] = eigvals
                pca_dict['pc_scores_' + key] = pc_scores[:,0:number_components_align]

        # Save 
        with open(os.path.join(path_pca_sub, 'pca_' + trial_type + '.pkl'), 'wb') as file:
            pickle.dump(pca_dict, file)


# Principal Component Analysis - incorrect trials

Compute PC scores of incorrect trials by projecting X_incorrect to eigvecs_correct

In [41]:
trial_type = 'incorrect_trials'

if trial_type == 'correct_trials':
    outputs = outputs_correct_trials
elif trial_type == 'incorrect_trials':
    outputs = outputs_incorrect_trials

for time_window in time_windows:

    if time_window == 'encode':

        segments = range(-5,8) # from [-500 -100] ms, to [800 1200] relative to onset of encode period

    elif time_window == 'maint':

        segments = range(-5,20) # from [-500 -100] ms, to [1800 2200] relative to onset of maint period

    elif time_window == 's2':

        segments = range(-5,2) # from [-500 -100] ms, to [200 600] relative to onset of s2 period

    for sub_i in subjects:

        # paths
        path_X_sub = os.path.join(path_root, 'results', folder_input, time_window + '_time_resolved', sub_i)
        path_pca_sub = os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)

        if not os.path.isdir(path_pca_sub):
            os.makedirs(path_pca_sub)

        # load X matrices
        filename = os.path.join(path_X_sub, 'X_dict_' + trial_type + '.pkl')
        with open(filename, 'rb') as file:
            X_dict = pickle.load(file)

        # load eigvecs of correct trials
        path_pca_sub = os.path.join(path_root, 'results', pca_folder, time_window + '_time_resolved', sub_i)
        filename = os.path.join(path_pca_sub, 'pca_correct_trials.pkl')
        with open(filename, 'rb') as file:
            eigvecs_correct_trials = pickle.load(file)

        pca_dict = {}

        # loop over outputs_orientation                    
        for output in outputs:

            for time_segment in segments:
                                
                    key = time_window + '_segment' + str(time_segment) + '_' + output

                    # store
                    pc_scores = X_dict[key] @ eigvecs_correct_trials['eigvecs_' + key][:,0:number_components_align]
                    pca_dict['pc_scores_' + key] = pc_scores[:,0:number_components_align]

        # Save the dictionary to a file using pickle
        with open(os.path.join(path_pca_sub, 'pca_' + trial_type + '.pkl'), 'wb') as file:
            pickle.dump(pca_dict, file)
